# Agent Identity SDK: STS Token Usage Example

This notebook demonstrates how to interactively use the `agent-identity-dev-sdk` to automatically manage workload access tokens and seamlessly inject STS credentials into decorated functions.

### Prerequisites

Before running the remaining cells, ensure you have configured your **Huawei Cloud** credentials (e.g. via Provider Chain or environment variables) and the desired region. You can set them here or rely on your environment.

In [ ]:
import os
import json
import uuid
from typing import Optional
from huaweicloudsdkiam.v5 import (
    IamClient,
    CreateAgencyV5Request,
    CreateAgencyReqBody,
)
from huaweicloudsdkagentidentity.v1 import AuthorizerType
from agentarts.sdk import (
    AgentArtsRuntimeContext,
    require_sts_token,
    IdentityClient,
)
from agentarts.sdk.identity.types import StsCredentials
from huaweicloudsdkcore.http.http_config import HttpConfig

# Manually set your credentials here for testing if not already in your environment.
# Ensure you DO NOT commit these secrets to version control.


# os.environ["HUAWEICLOUD_SDK_AK"] = "your-access-key-id"
# os.environ["HUAWEICLOUD_SDK_SK"] = "your-secret-access-key"
# os.environ["HUAWEICLOUD_SDK_REGION"] = "ap-southeast-4"
# os.environ["HUAWEICLOUD_SDK_PROJECT_ID"] = "your-project-id"
# os.environ["HUAWEICLOUD_SDK_DOMAIN_ID"] = "your-domain-id"

print("Environment variables ready.")

### Step 1: Set the Context

The SDK relies on contextual information (like `user_id`) being set before operations occur.
Here, we generate unique identifiers for this session and seed our `AgentIdentityContext`.

In [ ]:
# 0. Generate random identifiers for testing
random_suffix = uuid.uuid4().hex[:8]
user_id = f"user-{random_suffix}"
workload_name = f"sts-workload-{random_suffix}"

# 1. Set the user ID for context
AgentArtsRuntimeContext.set_user_id(user_id)
print(f"Context set for User ID: {user_id}")

### Step 2: Initialize the Identity Client

We construct the `IdentityClient` that will be used to interact with Huawei Cloud services.

In [ ]:
# 2. Initialize IdentityClient using the region from the environment
region = os.getenv("HUAWEICLOUD_SDK_REGION", "ap-southeast-4")
client = IdentityClient(region=region, ignore_ssl_verification=True)
print(f"IdentityClient initialized for region: {region}")

### Step 3: Create an IAM Agency

Before we can use an STS credential provider, we need an actual IAM agency in Huawei Cloud.
We'll use the Huawei Cloud IAM SDK (unionsdk) to dynamically create one for testing.

In [ ]:
# 3. Create an IAM Agency
region = os.getenv("HUAWEICLOUD_SDK_REGION", "ap-southeast-4")

print("Initializing IAM SDK Client to create a dynamic agency...")
hc = HttpConfig.get_default_config()
hc.ignore_ssl_verification = True
iam_client = (
    IamClient.new_builder()
    .with_http_config(hc)
    .with_endpoint("https://iam.myhuaweicloud.com")
    .build()
)

try:
    # Ensure your domain ID is available via env or provider chain
    domain_id = os.getenv("HUAWEICLOUD_SDK_DOMAIN_ID", "your-domain-id")

    agency_name = f"sts-agency-{random_suffix}"
    print(f"Creating dynamic agency '{agency_name}' in domain '{domain_id}'...")

    policy = {
        "Version": "5.0",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["sts:agencies:assume", "sts::SetContext"],
                "Principal": {"Service": ["service.AgentIdentity"]},
            }
        ],
    }

    req_body = CreateAgencyReqBody(
        agency_name=agency_name,
        trust_policy=json.dumps(policy),
        description="Dynamic agency created by unionsdk for STS token example",
    )

    create_response = iam_client.create_agency_v5(CreateAgencyV5Request(body=req_body))

    agency_urn = create_response.agency.urn
    print(f"Successfully created agency.\nURN: {agency_urn}")

except Exception as e:
    print(f"Failed to create agency dynamically (are credentials configured?): {e}")
    print("Falling back to a dummy URN for testing purposes.")
    agency_urn = "urn:huaweicloud:iam::123456789:agency:my-agency"

### Step 4: Create STS Credential Provider

Now we can create the STS credential provider referencing the agency URN we just created.

In [ ]:
# 4. Create STS Credential Provider
provider_name = f"sts-iam-provider-{random_suffix}"

print(f"Ensuring STS Credential Provider '{provider_name}' exists...")
try:
    provider = client.create_sts_credential_provider(
        name=provider_name, agency_urn=agency_urn
    )
    print(f"Created Provider: {provider.name}")
except Exception as e:
    print(f"Note: Provider might already exist or creation failed: {e}")

### Step 5: Create a Workload Identity

Next, we register our uniquely named workload identity with the Identity service.

In [ ]:
# 5. Create a workload identity
print(f"Creating workload identity: {workload_name}...")
# 'authorizer_type' is now a mandatory parameter in the underlying SDK.
# We use AuthorizerType.NONE for simplicity in this example.
workload = client.create_workload_identity(
    name=workload_name, authorizer_type=AuthorizerType.NONE
)
print(f"Created Workload: {workload.name}")

### Step 6: Fetch Workload Access Token

Using our registered workload and user ID, we request an access token. The SDK needs this token to authorize fetching specific STS credentials.

In [ ]:
# 6. Get workload access token
print(f"Getting workload access token for {workload.name} and user {user_id}...")
token = client.create_workload_access_token(
    workload_name=workload.name, user_id=user_id
)
print("Workload access token successfully retrieved.")

### Step 7: Define the Target Function

Now we define our target function (`access_huawei_resource`).
We decorate this function with `@require_sts_token` so the SDK handles STS token injection automatically.

In [ ]:
@require_sts_token(
    provider_name=provider_name,
    agency_session_name="example-session",
    ignore_ssl_verification=True,
)
def access_huawei_resource(sts_credentials: Optional[StsCredentials] = None) -> None:
    """Accesses Huawei Cloud resource with STS credentials.

    Args:
        sts_credentials: The STS credentials injected automatically by the SDK.
    """
    if sts_credentials:
        print("Successfully received STS Credentials!")
        print(f"AK: {sts_credentials.access_key_id[:5]}... (truncated)")
        print(f"SK: {sts_credentials.secret_access_key[:5]}... (truncated)")
        print(f"SecurityToken length: {len(sts_credentials.security_token)}")
    else:
        print("No STS credentials received.")


print("Decorated function defined.")

### Step 8: Inject the Token and Call the Function

Finally, we place the retrieved token into our `AgentIdentityContext`.
When we call `access_huawei_resource()`, the `@require_sts_token` decorator will use this token automatically to assume the agency and pass the temporary credentials into the function.

In [ ]:
# 7. Set the retrieved token in the context
AgentArtsRuntimeContext.set_workload_access_token(token)

# 7. Call the decorated function
print("Calling the decorated function...")
access_huawei_resource()
print("Execution complete!")